# Day 3 | Lab 3.1: LangChain Fundamentals — Chains, Modern Memory, LCEL Composition

**Duration:** ~1.5 hours

**Scenario.** Customer-complaint pipeline (sentiment + categorization + reply drafting) — preserved from the GM source notebook. Domain-agnostic illustrative content; trainees focus on LCEL mechanics, not the customer-support business.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Compose LCEL chains with `prompt | llm | parser` and stream their output.
2. Use modern conversational memory via **`RunnableWithMessageHistory`** + `InMemoryChatMessageHistory` — the LangChain v1 replacement for the deprecated `ConversationBufferMemory` family.
3. Emit granular streaming events with **`.astream_events()`** for streaming UIs (foundation for the FastAPI deployment in Module 10).
4. Compose multi-step pipelines with `RunnableLambda` and `RunnablePassthrough`.
5. Run independent chain branches in parallel with **`RunnableParallel`** and measure the latency win.

**What we deliberately drop from the source notebook**
- The deprecated `ConversationBufferMemory` / `ConversationSummaryMemory` / `ConversationEntityMemory` section — replaced with `RunnableWithMessageHistory`.
- The `PydanticOutputParser` deep-dive — already covered in **Lab 2.4**.

---


## 1. Install Dependencies

In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-openai>=1.0' 'langchain-groq>=0.2' pydantic


## 2. API Key Configuration

In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Verify keys are loaded (prints status, never prints actual values)
for key in ['OPENAI_API_KEY', 'GROQ_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


## 3. Initialize LangChain LLM Wrappers

We'll use OpenAI and Groq side-by-side throughout this module — the exact same chain works on both, demonstrating the provider-agnostic value of LangChain abstractions.


In [3]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# Primary model — OpenAI gpt-4.1-mini (fast, cheap, supports temperature)
llm_openai = ChatOpenAI(model="gpt-4.1-mini", temperature=0.7)

# Open-source alternative — Llama 3.3 70B via Groq's LPU hardware
llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

# Quick smoke test — both should respond to a simple invoke
print("OpenAI:", llm_openai.invoke("Say 'hello' in one word.").content)
print("Groq:  ", llm_groq.invoke("Say 'hello' in one word.").content)
print("\nBoth LLM clients ready ✅")

OpenAI: Hello!
Groq:   Hello

Both LLM clients ready ✅


---

## 4. Basic LCEL Chains — Compose, Stream, Parse

The LCEL pipe (`|`) is sugar for `RunnableSequence`. Three runnables in a row → three steps in your pipeline. Stream the output, swap providers, replace the parser — each is a one-line change.


### 4a. Basic Chain with `StrOutputParser`

**Pattern:** `prompt | llm | parser`. The prompt template is rendered with input variables, the LLM produces a `BaseMessage`, and the parser extracts the string content.


In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define the prompt template with placeholders
prompt = ChatPromptTemplate.from_template(
    """You are a marketing copywriter for a retail bank.
Write a concise 3-sentence product summary for the following loan product.

Product: {product_name}
Target Audience: {audience}

Summary:"""
)

# Build the chain: prompt → LLM → string parser
chain = prompt | llm_openai | StrOutputParser()

# Invoke with input variables
result = chain.invoke({
    "product_name": "SmartHome Loan",
    "audience": "First-time homebuyers aged 25-35"
})

print("--- OpenAI Result ---")
print(result)
print(f"\nType: {type(result)}")  # Should be <class 'str'>

--- OpenAI Result ---
The SmartHome Loan is designed to help first-time homebuyers aged 25-35 secure their dream home with ease. Enjoy competitive interest rates, flexible payment options, and personalized support throughout your home-buying journey. Make your move smarter and simpler with the SmartHome Loan.

Type: <class 'langchain_core.messages.base.TextAccessor'>


In [5]:
# Same chain, swap to Groq — only the LLM changes, everything else stays the same
chain_groq = prompt | llm_groq | StrOutputParser()

result_groq = chain_groq.invoke({
    "product_name": "SmartHome Loan",
    "audience": "First-time homebuyers aged 25-35"
})

print("--- Groq (Llama 3.3) Result ---")
print(result_groq)

--- Groq (Llama 3.3) Result ---
The SmartHome Loan is designed to help first-time homebuyers achieve their dream of owning a home with a competitive interest rate and flexible repayment terms. With a low deposit requirement and no ongoing fees, this loan is perfect for young professionals and couples looking to enter the property market. By choosing the SmartHome Loan, you'll not only get a great deal on your home loan, but also access to expert guidance and support every step of the way to make your home ownership journey as smooth as possible.


### 4b. Structured Output with `JsonOutputParser`

When the response should be a JSON object (dict). Pair with `format_instructions` so the model knows the expected shape. Streamable and dict-typed.


In [6]:
from langchain_core.output_parsers import JsonOutputParser

# JsonOutputParser without a Pydantic model — returns a plain dict
json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_template(
    """Extract structured policy details from the following text.
Return a JSON object with keys: policy_type, coverage_amount, premium_monthly, deductible, term_years.

{format_instructions}

Text: {policy_text}"""
)

json_chain = json_prompt | llm_openai | json_parser

result = json_chain.invoke({
    "policy_text": """Our Gold Health Shield plan offers comprehensive coverage of up to $500,000
    with a monthly premium of $350 and an annual deductible of $1,500. The policy term is 5 years
    with guaranteed renewal.""",
    "format_instructions": json_parser.get_format_instructions()
})

print("Parsed JSON output:")
print(result)
print(f"\nType: {type(result)}")  # Should be <class 'dict'>
print(f"Coverage: {result.get('coverage_amount')}")

Parsed JSON output:
{'policy_type': 'Gold Health Shield', 'coverage_amount': 500000, 'premium_monthly': 350, 'deductible': 1500, 'term_years': 5}

Type: <class 'dict'>
Coverage: 500000


### 4c. Type-Safe Output (covered in Lab 2.4)

> The `PydanticOutputParser` deep-dive lives in **Lab 2.4 — LangChain Output Parsers Deep-Dive**. The modern path is `llm.with_structured_output(Model)` for schema-validated output. We don't repeat it here.


### 4d. Streaming with LCEL

Calling `.stream(...)` instead of `.invoke(...)` yields tokens as they arrive. Same chain, different terminal method.


In [15]:
# Reuse the basic chain from 4a — streaming just works
stream_chain = prompt | llm_openai | StrOutputParser()

print("Streaming response token by token:\n")
for chunk in stream_chain.stream({
    "product_name": "Green Auto Loan",
    "audience": "Eco-conscious consumers buying electric vehicles"
    }):
    print(chunk, end="", flush=True)

print("\n\n✅ Streaming complete")

Streaming response token by token:

Drive green with our Green Auto Loan, designed exclusively for eco-conscious buyers of electric vehicles. Enjoy competitive rates and flexible terms that make going electric affordable and easy. Turn your commitment to the environment into savings today.

✅ Streaming complete


### Key Takeaways — Task 1

| Component | Class | What It Does |
|-----------|-------|--------------|
| **Prompt Template** | `ChatPromptTemplate` | Structures the input with placeholders |
| **LLM** | `ChatOpenAI` / `ChatGroq` | Generates the response |
| **StrOutputParser** | `StrOutputParser` | Returns plain text string |
| **JsonOutputParser** | `JsonOutputParser` | Parses into Python dict |
| **PydanticOutputParser** | `PydanticOutputParser` | Parses into validated Pydantic object |

The LCEL pipe (`|`) operator connects these components — output of one feeds into the next.

---

## 5. Conversational Memory with `RunnableWithMessageHistory`

The deprecated `ConversationBufferMemory` / `ConversationSummaryMemory` / `ConversationEntityMemory` classes have been replaced in LangChain v1 with **`RunnableWithMessageHistory`**, which composes naturally with LCEL. The history store is a function of `session_id`, so the same chain can hold many parallel conversations.

**Why the old classes are out of favour**
- They live outside the LCEL world (no `.invoke`/`.stream`/`.batch` interface).
- They couple history *storage* and history *injection* into one class — you can't swap an in-memory store for a Postgres-backed one without rewriting.
- They don't compose with `RunnableParallel`, fallback, retries, or any other modern Runnable feature.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1) Build the underlying chain — accepts {input, history}
memory_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a friendly assistant. Remember what the user tells you.'),
    MessagesPlaceholder('history'),
    ('user', '{input}'),
])
memory_chain = memory_prompt | chatgpt   # chatgpt was initialized in section 3

# 2) Per-session history store — keyed by session_id
store: dict[str, InMemoryChatMessageHistory] = {}

def get_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 3) Wrap the chain with auto-history management
chain_with_history = RunnableWithMessageHistory(
    memory_chain,
    get_history,
    input_messages_key='input',
    history_messages_key='history',
)


In [ ]:
# 3-turn conversation — the model should remember 'Priya' across turns
config = {'configurable': {'session_id': 'user-priya-001'}}

r1 = chain_with_history.invoke({'input': 'Hi, my name is Priya and I like banking software.'}, config=config)
print('Turn 1 →', r1.content)

r2 = chain_with_history.invoke({'input': 'What is my name?'}, config=config)
print('Turn 2 →', r2.content)

r3 = chain_with_history.invoke({'input': 'Recommend a Python library I should learn given my interests.'}, config=config)
print('Turn 3 →', r3.content)


In [ ]:
# Inspect the history store — actual messages live here
history_obj = store['user-priya-001']
print(f'Total messages stored: {len(history_obj.messages)}\n')
for i, msg in enumerate(history_obj.messages, 1):
    print(f'{i}. [{type(msg).__name__}] {msg.content[:80]}...')


### 📝 Key Takeaways — Section 5
- **`RunnableWithMessageHistory`** is just another Runnable — composes with `.stream()`, `.batch()`, `.with_retry()`, etc.
- The `session_id` decoupling means **the same wrapped chain handles N parallel conversations** without leaking state between them.
- For production, swap `InMemoryChatMessageHistory` for `SQLChatMessageHistory`, `RedisChatMessageHistory`, or `PostgresChatMessageHistory` — same interface, persistent storage.
- Module 7 (LangGraph) goes further with `MemorySaver` / `SqliteSaver` / `PostgresSaver` for full graph-state checkpointing.


---

## 6. Streaming Events with `.astream_events()`

`.stream(...)` yields the final tokens. `.astream_events(...)` yields **every internal event** in the chain — model start/end, parser events, tool starts, retry events. Foundation for streaming UIs that show "thinking", "calling tool X", or per-step progress.

Used heavily in Module 10's FastAPI streaming endpoint.


In [ ]:
import asyncio

# Use the basic chain from §4a — any LCEL chain works
stream_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a concise banking copywriter.'),
    ('user', '{topic}'),
])
stream_chain = stream_prompt | chatgpt

async def consume_events(topic: str) -> None:
    async for event in stream_chain.astream_events({'topic': topic}, version='v2'):
        kind = event['event']
        # Show only the events most useful for a UI
        if kind == 'on_chat_model_start':
            print(f'[start] model={event["name"]}')
        elif kind == 'on_chat_model_stream':
            chunk = event['data']['chunk']
            if chunk.content:
                print(chunk.content, end='', flush=True)
        elif kind == 'on_chat_model_end':
            print('\n[end]')

await consume_events('Write a 2-sentence tagline for a UPI payments app called PayZap.')


In [ ]:
# Inspecting all event types — useful for understanding what's available
seen_events = set()
async for event in stream_chain.astream_events({'topic': 'Write a haiku about banking.'}, version='v2'):
    seen_events.add(event['event'])

print('Event types emitted:')
for e in sorted(seen_events):
    print(f'  • {e}')


### 📝 When to use which streaming method

| Method | Returns | Use when |
|---|---|---|
| `.invoke()` | Final result, blocking | Backend services, batch pipelines, agents |
| `.stream()` | Final-output tokens, one chunk at a time | User-facing chat where you want to show the answer streaming in |
| `.astream_events()` | All internal events (start, stream, end, tool calls, etc.) | Streaming UIs that show *intermediate* state — "thinking", "calling tool X", per-step progress |

Module 10's FastAPI endpoint uses `.astream_events()` to power a Server-Sent-Events (SSE) stream.


---

## 7. Multi-Step Composition with `RunnableLambda`

Real pipelines have post-processing between steps — extracting a field, calling a deterministic function, reformatting. `RunnableLambda` is the escape hatch from LCEL into ordinary Python without breaking the pipe syntax.


In [16]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableLambda

# --- Step 1: Extract structured details from raw complaint ---
extract_prompt = ChatPromptTemplate.from_template(
    """Extract key details from this customer complaint. Return ONLY valid JSON with these keys:
- customer_name
- issue_type (one of: billing, claim_denial, service_delay, coverage_question, other)
- severity (one of: low, medium, high, critical)
- product_mentioned
- brief_summary (one sentence)

Complaint: {complaint}"""
)

extract_chain = extract_prompt | llm_openai | JsonOutputParser()

# --- Step 2: Transform into internal ticket format ---
transform_prompt = ChatPromptTemplate.from_template(
    """Given these extracted complaint details, generate an internal support ticket.
Return ONLY valid JSON with these keys:
- ticket_id (format: HLTH-XXXX where X is a random digit)
- priority (map severity: critical→P1, high→P2, medium→P3, low→P4)
- department (map issue_type: billing→Finance, claim_denial→Claims, service_delay→Operations, coverage_question→Underwriting, other→General)
- sla_hours (P1=4, P2=8, P3=24, P4=48)
- customer_name
- summary

Extracted details: {extracted}"""
)

transform_chain = transform_prompt | llm_openai | JsonOutputParser()

# --- Step 3: Generate acknowledgment email ---
email_prompt = ChatPromptTemplate.from_template(
    """Write a brief, empathetic acknowledgment email to the customer based on this support ticket.
Include the ticket ID, expected resolution time, and assigned department.
Keep it professional and under 100 words.

Ticket details: {ticket}"""
)

email_chain = email_prompt | llm_openai | StrOutputParser()

In [17]:
# --- Run the full pipeline on a sample complaint ---
sample_complaint = """Dear Sir/Madam,
I am Rajesh Sharma and I have been a loyal customer of your HealthGuard Premium plan for 3 years.
Last month, my wife was hospitalized for an emergency surgery. I submitted the claim (Ref: CLM-78234)
two weeks ago but have received NO response. When I called your helpline, I was put on hold for
45 minutes and then disconnected. This is absolutely unacceptable for a premium plan customer.
The hospital is now sending me payment reminders. Please resolve this URGENTLY.
Regards, Rajesh Sharma"""

# Step 1: Extract
print("📋 Step 1 — Extracting details...")
extracted = extract_chain.invoke({"complaint": sample_complaint})
print(f"Extracted: {extracted}")

# Step 2: Transform
print("\n🔄 Step 2 — Creating internal ticket...")
ticket = transform_chain.invoke({"extracted": str(extracted)})
print(f"Ticket: {ticket}")

# Step 3: Generate email
print("\n📧 Step 3 — Generating acknowledgment email...")
email = email_chain.invoke({"ticket": str(ticket)})
print(f"Email:\n{email}")

📋 Step 1 — Extracting details...
Extracted: {'customer_name': 'Rajesh Sharma', 'issue_type': 'service_delay', 'severity': 'critical', 'product_mentioned': 'HealthGuard Premium plan', 'brief_summary': "Rajesh Sharma's emergency surgery claim has not been responded to for two weeks, with poor customer service and urgent payment reminders from the hospital."}

🔄 Step 2 — Creating internal ticket...
Ticket: {'ticket_id': 'HLTH-4827', 'priority': 'P1', 'department': 'Operations', 'sla_hours': 4, 'customer_name': 'Rajesh Sharma', 'summary': "Rajesh Sharma's emergency surgery claim under HealthGuard Premium plan has not been responded to for two weeks, accompanied by poor customer service and urgent payment reminders from the hospital."}

📧 Step 3 — Generating acknowledgment email...
Email:
Subject: Acknowledgment of Your Support Ticket HLTH-4827

Dear Mr. Sharma,

Thank you for reaching out regarding your emergency surgery claim. We sincerely apologize for the delay and the inconvenience cau

In [18]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def run_full_pipeline(complaint_text):
    """End-to-end complaint processing: extract → transform → generate email."""
    # Step 1
    extracted = extract_chain.invoke({"complaint": complaint_text})
    # Step 2
    ticket = transform_chain.invoke({"extracted": str(extracted)})
    # Step 3
    email = email_chain.invoke({"ticket": str(ticket)})

    return {
        "extracted": extracted,
        "ticket": ticket,
        "email": email
    }

# Wrap as a Runnable so it integrates with LCEL ecosystem
pipeline = RunnableLambda(run_full_pipeline)

# Test with a different complaint
new_complaint = """Hi, I'm Meena Patel. I noticed I was charged ₹2,400 for a wellness checkup
that should be covered under my Silver Health plan's preventive care benefits.
Can you please look into this billing error? My policy number is SH-90211. Thanks."""

result = pipeline.invoke(new_complaint)

print("🎯 Full pipeline result:")
print(f"\n📋 Extracted: {result['extracted']}")
print(f"\n🎫 Ticket: {result['ticket']}")
print(f"\n📧 Email:\n{result['email']}")

🎯 Full pipeline result:

📋 Extracted: {'customer_name': 'Meena Patel', 'issue_type': 'billing', 'severity': 'medium', 'product_mentioned': 'Silver Health plan', 'brief_summary': "Customer was charged ₹2,400 for a wellness checkup that should have been covered under the Silver Health plan's preventive care benefits."}

🎫 Ticket: {'ticket_id': 'HLTH-4837', 'priority': 'P3', 'department': 'Finance', 'sla_hours': 24, 'customer_name': 'Meena Patel', 'summary': "Customer was charged ₹2,400 for a wellness checkup that should have been covered under the Silver Health plan's preventive care benefits."}

📧 Email:
Subject: Acknowledgment of Your Support Ticket HLTH-4837

Dear Meena Patel,

Thank you for reaching out regarding the charge of ₹2,400 for your wellness checkup. We understand your concern and are reviewing the matter with our Finance team. Your ticket (HLTH-4837) has been assigned to the Finance department, and we aim to resolve this within 24 hours.

We appreciate your patience and wi

---

## 8. Parallel Branches with `RunnableParallel`

When chain branches are independent (e.g., classification + sentiment + summary on the same input), run them concurrently. `RunnableParallel` schedules them with `asyncio.gather` under the hood — typical 2–4× wall-clock speedup for I/O-bound LLM calls.


In [19]:
from langchain_core.runnables import RunnableParallel

product_description = """NutriVita Pro — Advanced probiotic supplement with 50 billion CFU,
15 clinically studied strains, delayed-release capsules for maximum gut absorption.
Supports digestive health, immune function, and mental clarity. Doctor-recommended.
Now 30% off for new customers. 90-day money-back guarantee."""

# Three parallel analysis chains
marketing_chain = ChatPromptTemplate.from_template(
    "Rate this product description's marketing appeal (1-10) and suggest one improvement. "
    "Be brief.\n\nDescription: {text}"
) | llm_openai | StrOutputParser()

compliance_chain = ChatPromptTemplate.from_template(
    "Check this product description for potential regulatory compliance issues "
    "(FDA, FTC advertising rules). List any concerns briefly.\n\nDescription: {text}"
) | llm_openai | StrOutputParser()

sentiment_chain = ChatPromptTemplate.from_template(
    "Analyze the tone and sentiment of this product description. "
    "Is it trustworthy or hype-heavy? One paragraph.\n\nDescription: {text}"
) | llm_groq | StrOutputParser()  # Using Groq for variety

# Run all three in parallel
parallel_analysis = RunnableParallel(
    marketing=marketing_chain,
    compliance=compliance_chain,
    sentiment=sentiment_chain
)

results = parallel_analysis.invoke({"text": product_description})

print("🔀 Parallel Analysis Results:\n")
for key, value in results.items():
    print(f"{'─' * 50}")
    print(f"📊 {key.upper()}")
    print(f"{'─' * 50}")
    print(value)
    print()

🔀 Parallel Analysis Results:

──────────────────────────────────────────────────
📊 MARKETING
──────────────────────────────────────────────────
Rating: 8

Improvement: Add a relatable benefit or lifestyle outcome to make it more emotionally engaging, e.g., "Feel energized and balanced every day."

──────────────────────────────────────────────────
📊 COMPLIANCE
──────────────────────────────────────────────────
Here are potential regulatory compliance concerns with the product description:

1. **Structure/Function Claims:**  
   - "Supports digestive health, immune function, and mental clarity" are structure/function claims that require proper substantiation and a disclaimer such as:  
   *“These statements have not been evaluated by the FDA. This product is not intended to diagnose, treat, cure or prevent any disease.”*  
   - If this disclaimer is missing, it may violate FDA rules.

2. **"Doctor-recommended":**  
   - This claim implies endorsement and must be truthful and substantiat

In [ ]:
import time

# Shared prompt — works with any LLM
summary_prompt = ChatPromptTemplate.from_template(
    """Summarize this financial news in exactly 2 sentences for a retail investor audience.

News: {news}"""
)

news_text = """The Reserve Bank of India kept the repo rate unchanged at 6.5% for the eighth
consecutive time, citing persistent food inflation concerns. However, the central bank revised
its GDP growth forecast for FY25 upward to 7.2% from the earlier 7.0%, signaling confidence
in India's economic momentum despite global uncertainties."""

# Test with both providers
for name, llm in [("OpenAI (gpt-4.1-mini)", llm_openai), ("Groq (Llama 3.3 70B)", llm_groq)]:
    chain = summary_prompt | llm | StrOutputParser()
    start = time.time()
    result = chain.invoke({"news": news_text})
    latency = round(time.time() - start, 2)
    print(f"\n🤖 {name} ({latency}s):")
    print(f"   {result}")

---

## 9. Conclusion & Key Takeaways

| Concept | What you should walk away with |
|---|---|
| **LCEL pipe** | `prompt \| llm \| parser` is a `RunnableSequence` — fully composable, streamable, batchable |
| **Modern memory** | `RunnableWithMessageHistory` + `InMemoryChatMessageHistory` is the v1 pattern; old `Memory` classes are deprecated |
| **Per-session state** | History is keyed by `session_id` — same chain handles N parallel conversations |
| **`.astream_events()`** | Yields all internal events — for streaming UIs that show progress, not just final tokens |
| **`RunnableLambda`** | Escape hatch from LCEL into ordinary Python without breaking the pipe |
| **`RunnableParallel`** | Independent branches run concurrently — measurable latency win for multi-aspect analysis |

## 10. Stretch Exercise (Optional)
1. **Persistent history.** Replace `InMemoryChatMessageHistory` with `SQLChatMessageHistory` (SQLite) and verify history survives a kernel restart.
2. **Token budget guard.** Add a `RunnableLambda` after `get_history` that trims the message history to the last 10 turns when it grows too long.
3. **Streaming UI mock.** Use `.astream_events()` to print a status line ("⏳ thinking…") on `on_chat_model_start` and clear it on `on_chat_model_stream`.
4. **Memory + parallel.** Fan out the user's message into 3 parallel branches (sentiment, intent, summary) using `RunnableParallel`, then store the synthesized result in conversation history.
5. **Per-user system prompt.** Plumb a `user_role` config value through `chain_with_history.invoke(..., config={...})` and have the system prompt vary by role (`compliance_officer`, `customer`, `analyst`).

---

**Next Lab:** Lab 3.2 — Tool Calling Agents (LangChain v1) 🛠️


---

## Interview Preparation

The questions below mirror what client interviewers commonly ask about LangChain fundamentals. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. Why is `ConversationBufferMemory` deprecated, and what does the new pattern do better?**

*Hint:* Think LCEL composition + storage decoupling.

*Answer sketch:* The old `Memory` classes lived outside LCEL — no `.invoke`/`.stream`/`.batch` interface and they coupled storage with injection. `RunnableWithMessageHistory` is itself a Runnable, composes with every other LCEL primitive (parallel, fallback, retry, batch), and the storage is a swappable function (in-memory → SQLite → Postgres) without rewriting the chain.

---

**Q2. How is `RunnableWithMessageHistory` architecturally different from the old `Memory` classes?**

*Hint:* Wrapper vs embedded class.

*Answer sketch:* `RunnableWithMessageHistory` is a **wrapper** around any Runnable that adds history-injection at the input boundary and history-update at the output boundary. The wrapped chain stays pure (no internal state). Old `Memory` was an embedded subclass that mutated chain state — fundamentally incompatible with stateless distributed deployment.

---

**Q3. What's a `session_id` and why does the history store need one?**

*Hint:* Multi-tenant chat.

*Answer sketch:* A `session_id` is the key under which conversation history is stored. One `RunnableWithMessageHistory` instance can serve thousands of parallel conversations — each user/session gets its own history bucket. Without a session key, all conversations would mix.

---

**Q4. LCEL pipe syntax — what does `prompt | llm | parser` actually compile to internally?**

*Hint:* `RunnableSequence`.

*Answer sketch:* The `|` operator is sugar for `RunnableSequence(prompt, llm, parser)`. The sequence's `.invoke({...})` calls each step's `.invoke` in order, passing the previous output to the next. Same applies to `.stream`, `.batch`, `.ainvoke` — the sequence delegates to each step.

---

**Q5. When should you use `.stream()` vs `.astream_events()`?**

*Hint:* Final tokens vs all internal events.

*Answer sketch:* `.stream()` yields the chain's *final* output as it's produced — perfect for chat UIs that just need the answer streamed. `.astream_events()` yields *every* internal event (model start/end, tool calls, parser events) — needed when the UI shows intermediate state like "thinking" or "calling tool X". Module 10's FastAPI endpoint uses the latter.

---

**Q6. How do you persist chat history across server restarts?**

*Hint:* Different `*ChatMessageHistory` implementations.

*Answer sketch:* Replace `InMemoryChatMessageHistory` with `SQLChatMessageHistory` (SQLite/Postgres-backed), `RedisChatMessageHistory`, `DynamoDBChatMessageHistory`, etc. The `RunnableWithMessageHistory` wrapper stays unchanged — only the `get_history(session_id)` factory points to a persistent store.

---

**Q7. What is the difference between a `Runnable` and a `Chain` in LangChain?**

*Hint:* The `Chain` base class is legacy.

*Answer sketch:* `Runnable` is the modern (LCEL-era) abstract interface — anything with `.invoke`, `.stream`, `.batch`, `.ainvoke`, `.astream`, `.abatch`. `Chain` was the legacy class hierarchy (`LLMChain`, `SimpleSequentialChain`, etc.) — embedded multiple concerns in one class, no streaming, no async. New code should always reach for `Runnable`-based composition.

---

**Q8. Why does the team use Pydantic for input/output schemas in LCEL chains?**

*Hint:* Type safety + provider-validated schemas.

*Answer sketch:* Pydantic gives runtime validation, IDE autocomplete on `result.field_name`, and a JSON-schema export consumable by every modern LLM provider's structured-output API. `llm.with_structured_output(MyModel)` (Lab 2.4) routes the schema through the provider so the model returns ONLY valid JSON conforming to your schema — eliminating the parse-failure handling that prompt-only JSON requires.
